# 静态 Shape 执行优化技术

4.2 节我们知道静态 Shape 模型已经能享受整图下沉。但「能跑」不等于「跑得最好」——在下沉的基础上，静态 Shape 还能从**内存**和**并发**两个维度进一步提升执行效率。本节从用户视角讲清：哪些优化是 GE 默认做的（了解即可）、哪些是用户可配置开启的（重点掌握）、以及怎么验证收益。

本节学习大纲如下：

- 优化全景：默认开启 vs 用户可配置
- 内存优化（一）：零拷贝输入输出
- 内存优化（二）：内存复用与冲突防护
- 计算图瘦身：TensorMove 消除、Concat No Task
- 多 stream 并发执行
- 收益验证：profiling 看内存与并发
- 小结

## 1. 优化全景：默认开启 vs 用户可配置

静态 Shape 的执行优化分两类，理解这个分界能让你少走弯路：

| 类别 | 代表优化 | 用户动作 |
| --- | --- | --- |
| **GE 默认/编译期自动** | 内存复用与编排、内存冲突防护、TensorMove 消除、Concat No Task、变量去重复用 | 无需配置，编译时自动生效，**了解即可** |
| **用户可配置** | 模型 I/O 地址/内存策略（`ACL_MDL_WORKSPACE_MEM_OPTIMIZE`、Device Tensor、`LoweringOption::always_zero_copy`）、流配置（在线 GE、Ascend Graph C++ 构建与 ATC 使用不同入口） | 根据执行接口和图编译入口选择对应的公开 option / 属性，**重点掌握** |

<p align="left"><img src="./images/static_optimization_overview.svg" alt="静态 Shape 执行优化全景" width="85%"></p>

> 思路：先确认当前执行接口属于 ACL 模型加载、Session Device I/O 还是 RT2 Lowering，再选择对应的 I/O 内存策略；流配置还要区分在线 GE、Ascend Graph C++ 构建与 ATC 入口。所有收益都应通过 profiling 对照验证。

## 2. 内存优化（一）：零拷贝输入输出

### 2.1 问题：执行路径中的额外 I/O 搬运

调用方为模型准备的 I/O 通常是 Device 缓冲区。如果模型 I/O 地址不能直接映射到这些缓冲区，执行路径还要在调用方缓冲区和模型内部内存之间做 D2D 搬运；若数据来自或返回 Host，应用边界上还会有 H2D/D2H。这些额外搬运可能成为推理延迟的重要来源。

```
 非零拷贝:  调用方 Device 输入buf ─D2D─▶ 模型输入内存 ─▶ 计算 ─▶ 模型输出内存 ─D2D─▶ 调用方 Device 输出buf
 零拷贝  :  模型直接使用调用方提供的 Device I/O 地址，省去模型侧中间拷贝
```

### 2.2 零拷贝的两个方向

| 方向 | 含义 |
| --- | --- |
| 输入零拷贝 | 模型直接读取调用方 Device 输入地址，避免模型侧 D2D 中间拷贝 |
| 输出零拷贝 | 模型结果写入调用方预分配的 Device 输出地址，避免模型侧 D2D 中间拷贝 |

在满足能力和内存约束时，GE 可为 Data / NetOutput 建立模型 I/O 地址映射，执行时刷新任务参数中的地址而不搬运 Tensor 数据；若某个 I/O 不支持地址刷新，运行时仍可能回退为显式拷贝。

### 2.3 用户怎么用零拷贝

不同接口的配置含义不能混用：

| 执行路径 | 用户入口 | 作用边界 |
| --- | --- | --- |
| ACL 离线模型加载（V1） | `ACL_MDL_WORKSPACE_MEM_OPTIMIZE` | 让工作内存不包含模型输入/输出区；ACL 内部会转换为 `ge.exec.reuseZeroCopyMemory` 图选项 |
| C++ Session 指定 stream | `RunGraphWithStreamAsync` + `kPlacementDevice` Tensor | 把调用方 Device 地址交给静态模型执行路径；是否存在回退拷贝仍要 profiling 验证 |
| RT2 StreamExecutor | `LoweringOption::always_zero_copy` | 强制输出零拷贝；外部输出内存不合法时直接报错，不再回退 |

```cpp
// ACL 模型加载：使用公开的 aclmdlConfigAttr，而不是直接设置内部图 option 字符串。
size_t policy = ACL_WORKSPACE_MEM_OPTIMIZE_INPUTOUTPUT;
aclmdlSetConfigOpt(config_handle, ACL_MDL_WORKSPACE_MEM_OPTIMIZE,
                   &policy, sizeof(policy));

// RT2：保留 unique_ptr 的所有权；这里只开启强输出零拷贝。
gert::LoweringOption opt;
opt.always_zero_copy = true;
auto exec = gert::LoadStreamExecutorFromModelData(model_data, opt, ret);
if (exec == nullptr) {
  // 处理加载失败
}
```

> **关键约束**：`always_zero_copy` 针对输出内存，调用方必须保证输出大小不小于按 shape 计算的 Tensor 大小且 placement 正确。`always_external_allocator` 是另一项独立承诺：只有在调用方能提供加载/执行所需的全部 allocator 时才能开启，否则会报错。零拷贝不会消除业务数据在 Host 与 Device 之间不可避免的传输；要省掉模型侧中间搬运，I/O 本身应是满足要求的 Device 缓冲区。

### 2.4 动手实践：把预分配的 Device 地址作为模型输出缓冲区

CANN 9.0 的 Python `ge.session` 只公开同步 `add_graph` / `run_graph`，不支持外部 Allocator 和异步 stream 接口；而 **C++ `Session::RunGraphWithStreamAsync`** 支持调用方传入 Device Tensor。因此本单元会自动编译并运行一个小型 C++ 样例，不需要手工执行任何命令。

样例先用 `aclrtMalloc` 为两个输入和一个输出分配 Device 内存，把这些地址封装成 `kPlacementDevice` Tensor，再异步执行静态 Add 图。验收条件是：**调用方预分配的地址仍作为输出缓冲区返回**，并且回拷到 Host 后结果为 `[11, 22, 33, 44, 55, 66]`。最后一次 D2H 仅用于教学数值校验，实际 Device 流水线可直接把该地址交给下游算子。

```cpp
output_desc.SetPlacement(ge::kPlacementDevice);
output.SetData(preallocated_device_addr, output_bytes, device_deleter);
session.RunGraphWithStreamAsync(graph_id, stream, inputs, outputs);
```

> 地址相同只能证明调用方缓冲区被保留为输出目的地址，**不能单独证明 Kernel 直接写入且没有内部 D2D 回退拷贝**。是否真正省掉数据搬运，应在第 6 节用 `--runtime-api=on` 检查模型执行窗口内的 memcpy 事件。

> **耗时提示**：单元包含一个小型 C++ 构建和首次 GE 在线编图，通常需要数十秒，具体取决于 Notebook CPU 与 CANN 环境；运行中只显示当前 1/2～2/2 步骤。


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)
# === 真机运行：自动构建 CANN 9.0 C++ 样例并验证 Device I/O 地址 ===
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

EXPECTED_REVISION = "04.04-cann9-device-io-v2"
source_dir = Path("Sources/04.04/zero_copy_device").resolve()
revision_file = source_dir / "SAMPLE_REVISION"
actual_revision = (
    revision_file.read_text(encoding="utf-8").strip()
    if revision_file.is_file()
    else "<missing>"
)
if actual_revision != EXPECTED_REVISION:
    raise RuntimeError(
        "4.04 样例源码不完整或与 Notebook 不匹配。\n源码目录：{}\n检测标识：{}\n要求标识：{}\n"
        "请确认 Notebook 与 Sources/04.04/zero_copy_device 来自同一套教程文件。".format(
            source_dir, actual_revision, EXPECTED_REVISION
        )
    )
if not os.environ.get("ASCEND_HOME_PATH"):
    raise RuntimeError("ASCEND_HOME_PATH 未设置，请先加载 CANN 9.0 环境。")


def run_command(command, cwd, visible_prefixes=()):
    process = subprocess.Popen(
        command,
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        errors="replace",
        env=os.environ.copy(),
    )
    lines = []
    for line in process.stdout:
        lines.append(line)
        if visible_prefixes and line.startswith(visible_prefixes):
            print(line, end="", flush=True)
    return_code = process.wait()
    output = "".join(lines)
    if return_code != 0:
        print("[ERROR] 末尾日志：")
        print("".join(lines[-60:]))
        raise RuntimeError("命令执行失败，退出码={}：{}".format(return_code, command[0]))
    return output


with tempfile.TemporaryDirectory(prefix="ge_0404_") as workspace:
    sample_dir = Path(workspace) / "zero_copy_device"
    shutil.copytree(
        source_dir,
        sample_dir,
        ignore=shutil.ignore_patterns("build", "__pycache__", "*.pyc"),
    )
    build_dir = sample_dir / "build"

    print("[INFO] Step 1/2: 配置并构建 CANN 9.0 C++ Device I/O 样例", flush=True)
    run_command(
        ["cmake", "-S", str(sample_dir), "-B", str(build_dir), "-DCMAKE_BUILD_TYPE=Release"],
        sample_dir,
    )
    run_command(
        ["cmake", "--build", str(build_dir), "--parallel", "4"],
        sample_dir,
    )

    executable = build_dir / "ge_zero_copy_device"
    if not executable.is_file():
        raise RuntimeError("未生成样例可执行文件：{}".format(executable))

    print("[INFO] Step 2/2: 在线编图并在 0 号 NPU 上验证预分配输出地址", flush=True)
    output = run_command(
        [str(executable)],
        sample_dir,
        visible_prefixes=("[INFO] 正在", "[INFO] 地址一致", "预分配输出地址", "GE 返回输出地址", "输出：", "[OK]"),
    )
    if "[OK] 调用方预分配的 Device 地址已保留为模型输出缓冲区，NPU 数值校验通过" not in output:
        raise RuntimeError("样例未输出 Device 输出地址与数值验证成功标记。")


## 3. 内存优化（二）：内存复用与冲突防护（默认）

### 3.1 内存复用：静态 Shape 的天然优势

静态 Shape 下所有 Tensor 大小编译期已知，GE 可以做**模型级内存编排**：让生命周期不重叠的 Tensor 复用同一块物理内存，大幅降低显存占用。这是编译期自动完成的，无需配置。

此外变量管理还做了两类复用：

- **权重去重复用**：相同二进制内容的常量权重共享同一内存地址（按内容比较）。
- **逻辑地址映射**：OM 中变量引用逻辑地址，加载时映射到真实物理地址，使同一 OM 可跨设备加载。

### 3.2 冲突防护：复用的安全网（默认）

多个算子共享同一块内存（符号合并、Inplace、引用）能省显存，但有冲突风险。GE 在编译期与运行时建立了完整防护，对用户透明：

| 冲突类型 | 典型场景 | GE 处理 |
| --- | --- | --- |
| 语义读写冲突 | 一块输出同时被读算子和写算子消费 | 插入 Identity 隔离 |
| 内存布局冲突 | 共享同一符号的锚点内存属性不兼容 | 检测并隔离 |
| 子图地址隔离 | While/If/Case 子图内外共享地址 | Pass 隔离地址空间 |
| 多流内存生命周期冲突 | 跨流访问的内存被源流提前回收 | 运行时生命周期管理 |

> 用户视角：这些是「省内存的同时不出错」的保障，无需手工配置。若你手工写过 Inplace / 引用类自定义算子，需关注这类规则避免精度问题。

## 4. 计算图瘦身：TensorMove 消除与 Concat No Task（默认）

这两类优化在编译期自动「删掉不必要的内存搬运」，对延迟有直接收益，对用户透明。

### 4.1 TensorMove 消除

`TensorMove` 算子本质是一次设备内存拷贝（memcpy），用于隔离两段内存的生命周期。但当数据从源到消费点之间没有写冲突时，这次拷贝就是冗余的。GE 在 O3 优化级别识别并删除冗余 `TensorMove`：

<p align="center"><img src="./images/tensormove_elimination.svg" alt="TensorMove 消除" width="70%"></p>

框架适配器转换来的模型常携带大量冗余 `TensorMove`，逐层消除对 E2E 性能提升可观。

### 4.2 Concat No Task

`Concat` 的多个输入在内存中**连续排列**，是输出复用首输入地址、避免数据搬运的核心前提，但不是唯一条件。只有编译器同时确认属于支持的 `ConcatD` / `ConcatV2D` 类型，并且 Known Shape、拼接维度、地址对齐、输入来源等约束均满足时，才会把对应的 Concat 标记为「虚拟算子」：

```
 普通 Concat:  输入A/B/C ─▶ [Concat Task 启动 AI Core 搬运] ─▶ 输出
 Concat No Task: 输入A/B/C 已连续且满足其余约束 ─▶ 输出复用首输入地址（不下发 Task、不搬运）
```

典型受益场景：分布式训练中 AllGather 后接 Concat 拼接完整 batch。

> 用户视角：这两项随标准优化流水线自动执行。是否命中取决于具体图结构，应在 profiling / dump 图中确认冗余 TensorMove 或目标 Concat 的 Task 是否消失。

## 5. 多 stream 并发执行

### 5.1 流（Stream）是什么

Device 上的计算任务通过「流」组织调度——**同一条流内任务严格按序，不同流之间可并行**。流分配的质量直接影响执行效率：流太少无法发挥并行，流太多又带来过多同步（Event/Notify）和资源开销。

对静态 Shape，GE 默认走**多流模式**，编译期基于完整图拓扑做精细的多流分配与流复用（按引擎分流、依赖分流、AllReduce 与反向并行等），让无依赖的算子并行执行。这部分是自动的。

### 5.2 按图编译入口选择单流配置

单流模式是静态 Shape 的编译能力，但不同图编译入口使用不同的公开写法，不能把字符串 key、C++ 常量和 ATC 命令行参数混用：

| 图编译入口 | 开启单流的写法 | 传入位置 |
| --- | --- | --- |
| 在线 GE（Python/C++ Session） | `ge.enableSingleStream=true`；Python 图级 options 写为 `{"ge.enableSingleStream": "true"}` | `Session::AddGraph` / `session.add_graph` 的图级 options |
| Ascend Graph C++ 构建 | `{ge::ir_option::ENABLE_SINGLE_STREAM, "true"}` | `aclgrphBuildInitialize` 的 `global_options` |
| ATC 离线编译 | `--enable_single_stream=true` | `atc` 命令行 |

其他流相关公开配置包括：节点属性 `ge::public_attr::USER_STREAM_LABEL` 用于让相同标签的节点进入同一逻辑流；在线 GE 编译 option `ge.event=notify` 用于选择 Notify 作为跨流同步资源。

<p align="left"><img src="./images/stream_modes.svg" alt="单流与多流模式对比" width="85%"></p>

> 注意：无论通过上述哪种入口开启单流模式，都与 `ge::public_attr::USER_STREAM_LABEL` 互斥——单流模式下再给子图设置用户流标签，逻辑流分配会报参数错误（E10055）。源码中的 `STREAM_LABEL`、`PARALLEL_GROUP` 属于编译器内部属性，不应作为通用用户配置接口。

### 5.3 多模型 / 多流并发的注意事项

- 多个模型在不同 stream 上并发时，注意 **流与内存 allocator 的绑定关系**：一个 allocator 仅对应唯一 stream，流同步前内存不可归还、allocator 不可析构。
- 并发会放大「多流内存生命周期冲突」风险，GE 运行时有防护，但用户自管内存时要遵守「流同步后再释放」。
- 调试期可按当前图编译入口临时开启单流排除并发因素：在线 GE 使用 `ge.enableSingleStream=true`，Ascend Graph C++ 构建使用 `ge::ir_option::ENABLE_SINGLE_STREAM`，ATC 使用 `--enable_single_stream=true`。

## 6. 收益验证：profiling 看内存与并发

和执行流程一样，优化是否生效、收益多少，**靠 profiling 量化**。

### 6.1 采集

```shell
# 同时采集任务时间、Runtime API/内存拷贝、AI Core 访存带宽和 NPU 内存信息
msprof --application="./static_infer" \
       --output=/tmp/prof_opt \
       --task-time=on \
       --runtime-api=on \
       --aic-metrics=Memory \
       --sys-hardware-mem=on
```

`--runtime-api=on` 用于采集 Runtime API 与 memcpy 事件，是判断是否仍有 H2D/D2H/D2D 回退拷贝的关键；`--aic-metrics=Memory` 反映 AI Core 的 UB/L1/L2 访存带宽，并不等同于模型显存峰值。`--sys-hardware-mem=on` 用于采集 NPU 内存信息，具体输出能力与产品型号有关。

### 6.2 看什么

| 优化项 | profiling 观察点 | 期望现象 |
| --- | --- | --- |
| 零拷贝 | Runtime API / timeline 中模型执行区间内的 H2D、D2H、D2D memcpy | 与基线相比，不必要的输入/输出拷贝消失或显著减少；仅比较 Tensor 地址不足以证明零拷贝 |
| TensorMove 消除 | dump 图 / task 列表中的 TensorMove | 冗余 TensorMove 不再产生 Task |
| Concat No Task | Concat 对应的 AI Core Task | 该 Concat 不再下发计算 Task |
| 内存复用 | `npu_mem_*.csv` 等 NPU 内存采样结果 | 在相同模型、batch 与并发度下，对比基线确认显存峰值下降 |
| 多流并发 | 多条 stream 的 timeline | 无依赖算子在不同流上时间重叠（并行）|
| 单流 vs 多流 | E2E 耗时 | 多流通常更快；若多流更慢需查同步开销 |

```
 并发是否生效（看多条 stream 是否时间重叠）：
   stream0: ▮▮▮▮      ▮▮▮▮
   stream1:    ▮▮▮▮▮      ▮▮▮     ← 与 stream0 在时间上重叠 = 真并行
```

> 建议：先按所用执行接口配置零拷贝并用 profiling 确认拷贝条目是否减少，再评估多流；两类优化的实际收益都应以同负载基线下的 E2E、同步开销和内存数据为准。

## 7. 小结

- 静态 Shape 执行优化分**默认自动**（内存复用、冲突防护、TensorMove 消除、Concat No Task、变量去重）与**用户可配置**（零拷贝、单/多流）两类。
- **零拷贝**要区分执行路径：ACL V1 使用 `ACL_MDL_WORKSPACE_MEM_OPTIMIZE`，C++ Session 可传入 Device Tensor，RT2 强输出零拷贝使用 `LoweringOption::always_zero_copy`；是否仍有回退拷贝应通过 profiling 验证。
- **内存复用 + 冲突防护**让静态 Shape 在省显存的同时保证正确性，对用户透明。
- **TensorMove 消除 / Concat No Task** 在编译期删除冗余内存搬运，降低 E2E 延迟。
- **多流并发**默认由 GE 自动分流；开启单流时，在线 GE 使用 `ge.enableSingleStream`，Ascend Graph C++ 构建使用 `ge::ir_option::ENABLE_SINGLE_STREAM`，ATC 使用 `--enable_single_stream`；单流模式与用户流标签互斥。
- 所有优化都应用 **profiling 验证**：启用 Runtime API 看拷贝，采集 NPU 内存看峰值，并结合 Task 数、多流时间重叠与 E2E 耗时判断。

> 下一节进入动态 Shape 的优化——核心是用**动态分档**把 Host 开销压回静态优化的水平。

## 课后练习

完成下列题目自测，如有错误建议结合本节对应小节复盘。

1. （判断题）零拷贝的目的是消除调用方 I/O 缓冲区与模型内部内存之间不必要的 H2D、D2H 或 D2D 搬运。

2. （判断题）TensorMove 消除和 Concat No Task 需要用户手工配置选项才能开启。

3. （判断题）静态 Shape 开启单流串行执行时，在线 GE 图编译使用 `ge.enableSingleStream=true`，Ascend Graph C++ 构建向 `aclgrphBuildInitialize` 传入 `{ge::ir_option::ENABLE_SINGLE_STREAM, "true"}`，ATC 使用 `--enable_single_stream=true`。

4. （单选题）ACL 离线模型加载（V1）中，哪个公开的 `aclmdlConfigAttr` 用于配置 workspace 的输入输出内存优化？
    A. `ACL_MDL_PRIORITY_INT32`
    B. `ACL_MDL_WORKSPACE_MEM_OPTIMIZE`
    C. `ACL_MDL_LOAD_TYPE_SIZET`
    D. `ACL_MDL_WEIGHT_PATH_PTR`

5. （单选题）关于 Device 上的「流（Stream）」，以下说法正确的是？
    A. 同一条流内的任务可以乱序并行执行
    B. 同一条流内任务严格按序，不同流之间可以并行
    C. 不同流之间也必须严格按序
    D. 流只能有一条

6. （单选题）在其他类型、Shape、维度与对齐等约束也满足时，Concat No Task 的核心前提是？
    A. Concat 的输入数量大于 3
    B. Concat 的输入在内存中天然连续排列，可直接复用首输入地址
    C. 模型是动态 Shape
    D. 开启了 Tiling 下沉

7. （多选题）以下关于静态 Shape 执行优化的描述，哪些是正确的？
    A. 静态 Shape 下 GE 可做模型级内存复用，降低显存占用
    B. 多流并发让无依赖算子在不同流上并行，跨流处插入 Event/Notify 同步
    C. RT2 开启 `always_zero_copy` 强输出零拷贝后，用户必须保证输出内存大小与 placement 正确，否则报错
    D. 多流一定比单流快，流越多越好

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/04.04_answer.txt